In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """
    Find the customer-churn project directory.
    """

    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:

        expected_file = (
            candidate
            / "data"
            / "processed"
            / "telco_churn_model.csv"
        )

        if expected_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the customer-churn project root."
    )


project_root = find_project_root(Path.cwd())

models_directory = project_root / "models"

model_results_directory = (
    project_root
    / "reports"
    / "model_results"
)

explainability_directory = (
    project_root
    / "reports"
    / "explainability"
)

database_directory = (
    project_root
    / "database"
)

database_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", project_root)

In [ ]:
prediction_path = (
    model_results_directory
    / "all_customer_oos_predictions.csv"
)

model_metadata_path = (
    models_directory
    / "final_tuned_churn_model_metadata.json"
)

reason_codes_path = (
    explainability_directory
    / "customer_reason_codes.csv"
)

reason_summary_path = (
    explainability_directory
    / "customer_reason_summary.csv"
)

fairness_metrics_path = (
    explainability_directory
    / "fairness_group_metrics.csv"
)


required_paths = [
    prediction_path,
    model_metadata_path
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )


print("Required input files found.")

In [ ]:
all_predictions = pd.read_csv(
    prediction_path
)


with open(
    model_metadata_path,
    "r",
    encoding="utf-8"
) as file:

    model_metadata = json.load(file)


model_name = model_metadata.get(
    "model_name",
    "Unknown model"
)

probability_source = model_metadata.get(
    "probability_source",
    "Unknown"
)

analytical_threshold = float(
    model_metadata.get(
        "selected_threshold",
        0.50
    )
)


print("Prediction shape:", all_predictions.shape)

print("Model:", model_name)

print(
    "Probability source:",
    probability_source
)

print(
    "Current analytical threshold:",
    analytical_threshold
)

display(all_predictions.head())

In [ ]:
required_prediction_columns = {
    "customerid",
    "actual_churn",
    "churn_probability",
    "prediction_source"
}


missing_columns = (
    required_prediction_columns
    -
    set(all_predictions.columns)
)


if missing_columns:

    raise ValueError(
        f"Prediction file is missing: "
        f"{missing_columns}"
    )


assert all_predictions[
    "churn_probability"
].between(0, 1).all()


assert all_predictions[
    "actual_churn"
].isin([0, 1]).all()


assert all_predictions[
    "customerid"
].is_unique


print(
    "Prediction validation passed."
)

print(
    "\nPrediction sources:"
)

display(
    all_predictions[
        "prediction_source"
    ].value_counts()
)

In [ ]:
prediction_source_normalized = (
    all_predictions[
        "prediction_source"
    ]
    .astype(str)
    .str.lower()
)


oof_predictions = (
    all_predictions[
        prediction_source_normalized
        .str.contains("oof")
    ]
    .copy()
)


holdout_predictions = (
    all_predictions[
        prediction_source_normalized
        .str.contains("holdout")
    ]
    .copy()
)


if oof_predictions.empty:

    raise ValueError(
        "No Training OOF rows were found."
    )


if holdout_predictions.empty:

    raise ValueError(
        "No Test holdout rows were found."
    )


print(
    "OOF training rows:",
    len(oof_predictions)
)

print(
    "Holdout rows:",
    len(holdout_predictions)
)

print(
    "OOF churn rate:",
    round(
        oof_predictions[
            "actual_churn"
        ].mean(),
        4
    )
)

print(
    "Holdout churn rate:",
    round(
        holdout_predictions[
            "actual_churn"
        ].mean(),
        4
    )
)

In [ ]:
def evaluate_threshold(
    data,
    threshold,
    policy_name,
    dataset_name
):
    """
    Evaluate one probability threshold.
    """

    y_true = (
        data["actual_churn"]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        data["churn_probability"]
        .astype(float)
        .to_numpy()
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()


    actual_churn_rate = y_true.mean()

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )


    result = {
        "dataset": dataset_name,

        "policy": policy_name,

        "threshold": float(
            threshold
        ),

        "customers": int(
            len(data)
        ),

        "customers_flagged": int(
            predictions.sum()
        ),

        "contact_rate": float(
            predictions.mean()
        ),

        "actual_churn_rate": float(
            actual_churn_rate
        ),

        "accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predictions
            )
        ),

        "precision": precision,

        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "f2": fbeta_score(
            y_true,
            predictions,
            beta=2,
            zero_division=0
        ),

        "precision_lift": (
            precision / actual_churn_rate
            if actual_churn_rate > 0
            else np.nan
        ),

        "true_negatives": int(tn),

        "false_positives": int(fp),

        "false_negatives": int(fn),

        "true_positives": int(tp)
    }


    if "cltv" in data.columns:

        result[
            "flagged_cltv"
        ] = float(
            data.loc[
                predictions == 1,
                "cltv"
            ].sum()
        )


    return result

In [ ]:
def threshold_for_capacity(
    probabilities,
    capacity_rate
):
    """
    Find the probability threshold that selects
    approximately the highest-risk capacity_rate
    fraction of customers.
    """

    if not 0 < capacity_rate < 1:

        raise ValueError(
            "capacity_rate must be between 0 and 1."
        )


    return float(
        np.quantile(
            probabilities,
            1 - capacity_rate,
            method="higher"
        )
    )

In [ ]:
oof_y = (
    oof_predictions[
        "actual_churn"
    ]
    .astype(int)
    .to_numpy()
)


oof_probability = (
    oof_predictions[
        "churn_probability"
    ]
    .astype(float)
    .to_numpy()
)


precision_curve, recall_curve, pr_thresholds = (
    precision_recall_curve(
        oof_y,
        oof_probability
    )
)


recall_threshold_table = pd.DataFrame(
    {
        "threshold": pr_thresholds,

        "precision": (
            precision_curve[:-1]
        ),

        "recall": (
            recall_curve[:-1]
        )
    }
)


recall_90_candidates = (
    recall_threshold_table[
        recall_threshold_table[
            "recall"
        ] >= 0.90
    ]
)


if not recall_90_candidates.empty:

    recall_90_threshold = float(
        recall_90_candidates[
            "threshold"
        ].max()
    )

else:

    recall_90_threshold = (
        analytical_threshold
    )


print(
    "Highest threshold retaining "
    "at least 90% OOF recall:",
    round(recall_90_threshold, 4)
)

In [ ]:
capacity_rates = [
    0.10,
    0.20,
    0.30,
    0.40
]


candidate_policies = [
    {
        "policy": "Default 0.50",
        "threshold": 0.50
    },

    {
        "policy": "Analytical F2 threshold",
        "threshold": analytical_threshold
    },

    {
        "policy": "Minimum 90% OOF recall",
        "threshold": recall_90_threshold
    }
]


for capacity_rate in capacity_rates:

    capacity_threshold = (
        threshold_for_capacity(
            oof_probability,
            capacity_rate
        )
    )


    candidate_policies.append(
        {
            "policy": (
                f"Top {int(capacity_rate * 100)}% "
                "campaign capacity"
            ),

            "threshold": (
                capacity_threshold
            )
        }
    )


candidate_policy_df = pd.DataFrame(
    candidate_policies
)


display(
    candidate_policy_df.round(4)
)

In [ ]:
oof_policy_results = pd.DataFrame(
    [
        evaluate_threshold(
            data=oof_predictions,

            threshold=row["threshold"],

            policy_name=row["policy"],

            dataset_name="Training OOF"
        )

        for _, row
        in candidate_policy_df.iterrows()
    ]
)


oof_policy_results = (
    oof_policy_results
    .sort_values(
        "contact_rate"
    )
    .reset_index(drop=True)
)


display(
    oof_policy_results.round(4)
)

In [ ]:
display(
    oof_policy_results[
        [
            "policy",
            "threshold",
            "contact_rate",
            "precision",
            "recall",
            "f1",
            "f2",
            "precision_lift",
            "false_positives",
            "false_negatives"
        ]
    ].round(4)
)

In [ ]:
CAMPAIGN_CAPACITY_RATE = 0.30

HIGH_RISK_CAPACITY_RATE = 0.10


campaign_threshold = (
    threshold_for_capacity(
        oof_probability,
        CAMPAIGN_CAPACITY_RATE
    )
)


high_risk_threshold = (
    threshold_for_capacity(
        oof_probability,
        HIGH_RISK_CAPACITY_RATE
    )
)


print(
    "Analytical threshold:",
    round(analytical_threshold, 4)
)

print(
    "Campaign capacity:",
    f"{CAMPAIGN_CAPACITY_RATE:.0%}"
)

print(
    "Campaign threshold:",
    round(campaign_threshold, 4)
)

print(
    "High-risk capacity:",
    f"{HIGH_RISK_CAPACITY_RATE:.0%}"
)

print(
    "High-risk threshold:",
    round(high_risk_threshold, 4)
)


assert (
    high_risk_threshold
    >=
    campaign_threshold
)

In [ ]:
selected_policies = [
    {
        "policy": "Default classifier",
        "threshold": 0.50
    },

    {
        "policy": "Analytical high-recall",
        "threshold": analytical_threshold
    },

    {
        "policy": "Campaign capacity",
        "threshold": campaign_threshold
    },

    {
        "policy": "High-risk priority",
        "threshold": high_risk_threshold
    }
]
selected_policy_results = []


for policy in selected_policies:

    selected_policy_results.append(
        evaluate_threshold(
            data=oof_predictions,

            threshold=policy[
                "threshold"
            ],

            policy_name=policy[
                "policy"
            ],

            dataset_name="Training OOF"
        )
    )


    selected_policy_results.append(
        evaluate_threshold(
            data=holdout_predictions,

            threshold=policy[
                "threshold"
            ],

            policy_name=policy[
                "policy"
            ],

            dataset_name="Test holdout"
        )
    )


selected_policy_comparison = pd.DataFrame(
    selected_policy_results
)


display(
    selected_policy_comparison[
        [
            "dataset",
            "policy",
            "threshold",
            "contact_rate",
            "precision",
            "recall",
            "f1",
            "f2",
            "precision_lift",
            "false_positives",
            "false_negatives"
        ]
    ].round(4)
)

In [ ]:
plot_data = (
    oof_policy_results
    .sort_values(
        "contact_rate"
    )
)


plt.figure(figsize=(9, 6))


plt.plot(
    plot_data[
        "contact_rate"
    ],

    plot_data[
        "precision"
    ],

    marker="o",

    label="Precision"
)


plt.plot(
    plot_data[
        "contact_rate"
    ],

    plot_data[
        "recall"
    ],

    marker="o",

    label="Recall"
)


plt.plot(
    plot_data[
        "contact_rate"
    ],

    plot_data[
        "f2"
    ],

    marker="o",

    label="F2"
)


plt.axvline(
    CAMPAIGN_CAPACITY_RATE,

    linestyle="--",

    label=(
        f"Chosen capacity = "
        f"{CAMPAIGN_CAPACITY_RATE:.0%}"
    )
)


plt.xlabel(
    "Fraction of Customers Flagged"
)

plt.ylabel("Metric")

plt.title(
    "Campaign Capacity and Model Performance"
)

plt.ylim(0, 1)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
business_predictions = (
    all_predictions.copy()
)


business_predictions[
    "model_threshold"
] = analytical_threshold


business_predictions[
    "campaign_threshold"
] = campaign_threshold


business_predictions[
    "high_risk_threshold"
] = high_risk_threshold


business_predictions[
    "analytical_predicted_churn"
] = (
    business_predictions[
        "churn_probability"
    ]
    >=
    analytical_threshold
).astype(int)


business_predictions[
    "campaign_eligible"
] = (
    business_predictions[
        "churn_probability"
    ]
    >=
    campaign_threshold
).astype(int)


business_predictions[
    "high_priority_customer"
] = (
    business_predictions[
        "churn_probability"
    ]
    >=
    high_risk_threshold
).astype(int)

In [ ]:
business_predictions[
    "business_risk_segment"
] = np.select(
    [
        business_predictions[
            "churn_probability"
        ] >= high_risk_threshold,

        business_predictions[
            "churn_probability"
        ] >= campaign_threshold,

        business_predictions[
            "churn_probability"
        ] >= analytical_threshold
    ],

    [
        "High Risk",
        "Medium Risk",
        "Watchlist"
    ],

    default="Low Risk"
)

In [ ]:
if "priority_score" not in (
    business_predictions.columns
):

    if "cltv" in business_predictions.columns:

        business_predictions[
            "priority_score"
        ] = (
            business_predictions[
                "churn_probability"
            ]
            *
            business_predictions[
                "cltv"
            ]
        )

    else:

        business_predictions[
            "priority_score"
        ] = (
            business_predictions[
                "churn_probability"
            ]
        )

if "monthly_charges" in (
    business_predictions.columns
):

    business_predictions[
        "expected_monthly_value_at_risk"
    ] = (
        business_predictions[
            "churn_probability"
        ]
        *
        business_predictions[
            "monthly_charges"
        ]
    )
business_predictions[
    "campaign_rank"
] = pd.NA


eligible_indices = (
    business_predictions[
        business_predictions[
            "campaign_eligible"
        ] == 1
    ]
    .sort_values(
        [
            "priority_score",
            "churn_probability"
        ],

        ascending=False
    )
    .index
)


business_predictions.loc[
    eligible_indices,
    "campaign_rank"
] = np.arange(
    1,
    len(eligible_indices) + 1
)


business_predictions[
    "campaign_rank"
] = (
    business_predictions[
        "campaign_rank"
    ]
    .astype("Int64")
)


In [ ]:
campaign_queue_preview = (
    business_predictions[
        business_predictions[
            "campaign_eligible"
        ] == 1
    ]
    .sort_values(
        "campaign_rank"
    )
)


preview_columns = [
    column

    for column in [
        "campaign_rank",
        "customerid",
        "churn_probability",
        "business_risk_segment",
        "cltv",
        "priority_score",
        "monthly_charges",
        "contract",
        "tenure_months",
        "prediction_source"
    ]

    if column
    in campaign_queue_preview.columns
]


display(
    campaign_queue_preview[
        preview_columns
    ].head(25)
)

In [ ]:
business_risk_summary = (
    business_predictions
    .groupby(
        "business_risk_segment",
        observed=True
    )
    .agg(
        customers=(
            "customerid",
            "count"
        ),

        average_churn_probability=(
            "churn_probability",
            "mean"
        ),

        actual_churn_rate=(
            "actual_churn",
            "mean"
        ),

        total_priority_score=(
            "priority_score",
            "sum"
        )
    )
    .reset_index()
)


business_risk_summary[
    "customer_percentage"
] = (
    business_risk_summary[
        "customers"
    ]
    /
    len(business_predictions)
)


display(
    business_risk_summary.round(4)
)

In [ ]:
business_prediction_path = (
    model_results_directory
    / "all_customer_business_predictions.csv"
)


threshold_comparison_path = (
    model_results_directory
    / "business_threshold_comparison.csv"
)


threshold_metadata_path = (
    models_directory
    / "business_threshold_policy.json"
)


business_predictions.to_csv(
    business_prediction_path,
    index=False
)


selected_policy_comparison.to_csv(
    threshold_comparison_path,
    index=False
)
threshold_metadata = {
    "model_name": model_name,

    "probability_source": (
        probability_source
    ),

    "analytical_threshold": (
        analytical_threshold
    ),

    "analytical_threshold_purpose": (
        "Recall-focused F2 classification"
    ),

    "campaign_capacity_rate": (
        CAMPAIGN_CAPACITY_RATE
    ),

    "campaign_threshold": (
        campaign_threshold
    ),

    "high_risk_capacity_rate": (
        HIGH_RISK_CAPACITY_RATE
    ),

    "high_risk_threshold": (
        high_risk_threshold
    ),

    "threshold_selection_data": (
        "Training out-of-fold predictions only"
    ),

    "holdout_usage": (
        "Evaluation only"
    ),

    "risk_segments": {
        "Low Risk": (
            f"p < {analytical_threshold:.6f}"
        ),

        "Watchlist": (
            f"{analytical_threshold:.6f} "
            f"<= p < {campaign_threshold:.6f}"
        ),

        "Medium Risk": (
            f"{campaign_threshold:.6f} "
            f"<= p < {high_risk_threshold:.6f}"
        ),

        "High Risk": (
            f"p >= {high_risk_threshold:.6f}"
        )
    },

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    )
}


with open(
    threshold_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        threshold_metadata,
        file,
        indent=4
    )


print(
    "Business predictions saved to:"
)

print(business_prediction_path)

print(
    "\nThreshold metadata saved to:"
)

print(threshold_metadata_path)

##Part B — Build the SQLite database

In [ ]:
if reason_codes_path.exists():

    customer_reason_codes = pd.read_csv(
        reason_codes_path
    )

else:

    customer_reason_codes = pd.DataFrame()


if reason_summary_path.exists():

    customer_reason_summary = pd.read_csv(
        reason_summary_path
    )

else:

    customer_reason_summary = pd.DataFrame()


if fairness_metrics_path.exists():

    fairness_group_metrics = pd.read_csv(
        fairness_metrics_path
    )

else:

    fairness_group_metrics = pd.DataFrame()


print(
    "Reason-code rows:",
    len(customer_reason_codes)
)

print(
    "Reason-summary rows:",
    len(customer_reason_summary)
)

print(
    "Fairness rows:",
    len(fairness_group_metrics)
)

In [ ]:
def existing_columns(
    dataframe,
    requested_columns
):
    """
    Return only columns that exist.
    """

    return [
        column

        for column in requested_columns

        if column in dataframe.columns
    ]


def make_sqlite_safe(dataframe):
    """
    Convert data types into SQLite-friendly forms.
    """

    result = dataframe.copy()


    for column in result.columns:

        if isinstance(
            result[column].dtype,
            pd.CategoricalDtype
        ):

            result[column] = (
                result[column]
                .astype(str)
            )


        if pd.api.types.is_bool_dtype(
            result[column]
        ):

            result[column] = (
                result[column]
                .astype(int)
            )


    result = result.replace(
        [np.inf, -np.inf],
        np.nan
    )


    return result

In [ ]:
customer_columns = existing_columns(
    business_predictions,

    [
        "customerid",
        "gender",
        "senior_citizen",
        "partner",
        "dependents",
        "city",
        "zip_code",
        "latitude",
        "longitude",
        "country",
        "state"
    ]
)


customers_table = (
    business_predictions[
        customer_columns
    ]
    .drop_duplicates(
        subset=["customerid"]
    )
    .reset_index(drop=True)
)


customers_table[
    "customerid"
] = (
    customers_table[
        "customerid"
    ].astype(str)
)


assert customers_table[
    "customerid"
].is_unique


print(
    "Customers table:",
    customers_table.shape
)

display(customers_table.head())

In [ ]:
service_columns = existing_columns(
    business_predictions,

    [
        "customerid",
        "phone_service",
        "multiple_lines",
        "internet_service",
        "online_security",
        "online_backup",
        "device_protection",
        "tech_support",
        "streaming_tv",
        "streaming_movies"
    ]
)


services_table = (
    business_predictions[
        service_columns
    ]
    .drop_duplicates(
        subset=["customerid"]
    )
    .reset_index(drop=True)
)


print(
    "Services table:",
    services_table.shape
)

display(services_table.head())

In [ ]:
billing_columns = existing_columns(
    business_predictions,

    [
        "customerid",
        "tenure_months",
        "contract",
        "paperless_billing",
        "payment_method",
        "monthly_charges",
        "total_charges",
        "cltv"
    ]
)


billing_table = (
    business_predictions[
        billing_columns
    ]
    .drop_duplicates(
        subset=["customerid"]
    )
    .reset_index(drop=True)
)


print(
    "Billing table:",
    billing_table.shape
)

display(billing_table.head())

In [ ]:
prediction_columns = existing_columns(
    business_predictions,

    [
        "customerid",
        "actual_churn",
        "churn_probability",
        "prediction_source",
        "analytical_predicted_churn",
        "campaign_eligible",
        "high_priority_customer",
        "business_risk_segment",
        "model_threshold",
        "campaign_threshold",
        "high_risk_threshold",
        "priority_score",
        "expected_monthly_value_at_risk",
        "campaign_rank"
    ]
)


churn_predictions_table = (
    business_predictions[
        prediction_columns
    ]
    .copy()
)


MODEL_VERSION = (
    "xgboost_sigmoid_v1"
)


churn_predictions_table[
    "model_version"
] = MODEL_VERSION


churn_predictions_table[
    "scored_at_utc"
] = datetime.now(
    timezone.utc
).isoformat()


print(
    "Predictions table:",
    churn_predictions_table.shape
)

display(
    churn_predictions_table.head()
)

In [ ]:
test_metrics = model_metadata.get(
    "test_metrics",
    {}
)


model_registry_table = pd.DataFrame(
    [
        {
            "model_version": (
                MODEL_VERSION
            ),

            "model_name": (
                model_name
            ),

            "probability_source": (
                probability_source
            ),

            "analytical_threshold": (
                analytical_threshold
            ),

            "campaign_threshold": (
                campaign_threshold
            ),

            "high_risk_threshold": (
                high_risk_threshold
            ),

            "campaign_capacity_rate": (
                CAMPAIGN_CAPACITY_RATE
            ),

            "test_roc_auc": (
                test_metrics.get(
                    "roc_auc"
                )
            ),

            "test_pr_auc": (
                test_metrics.get(
                    "pr_auc"
                )
            ),

            "test_precision": (
                test_metrics.get(
                    "precision"
                )
            ),

            "test_recall": (
                test_metrics.get(
                    "recall"
                )
            ),

            "test_f1": (
                test_metrics.get(
                    "f1"
                )
            ),

            "test_f2": (
                test_metrics.get(
                    "f2"
                )
            ),

            "registered_at_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            )
        }
    ]
)


display(model_registry_table)

In [ ]:
sql_tables = {
    "customers": (
        make_sqlite_safe(
            customers_table
        )
    ),

    "customer_services": (
        make_sqlite_safe(
            services_table
        )
    ),

    "customer_billing": (
        make_sqlite_safe(
            billing_table
        )
    ),

    "churn_predictions": (
        make_sqlite_safe(
            churn_predictions_table
        )
    ),

    "model_registry": (
        make_sqlite_safe(
            model_registry_table
        )
    ),

    "threshold_policy_results": (
        make_sqlite_safe(
            selected_policy_comparison
        )
    ),

    "business_risk_summary": (
        make_sqlite_safe(
            business_risk_summary
        )
    )
}


if not customer_reason_codes.empty:

    sql_tables[
        "customer_explanations"
    ] = make_sqlite_safe(
        customer_reason_codes
    )


if not customer_reason_summary.empty:

    sql_tables[
        "customer_reason_summary"
    ] = make_sqlite_safe(
        customer_reason_summary
    )


if not fairness_group_metrics.empty:

    sql_tables[
        "fairness_group_metrics"
    ] = make_sqlite_safe(
        fairness_group_metrics
    )

In [ ]:
database_path = (
    database_directory
    / "customer_churn.db"
)


with sqlite3.connect(
    database_path
) as connection:

    connection.execute(
        "PRAGMA foreign_keys = ON;"
    )


    for table_name, dataframe in (
        sql_tables.items()
    ):

        dataframe.to_sql(
            table_name,

            connection,

            if_exists="replace",

            index=False
        )


        print(
            f"Loaded {len(dataframe):,} rows "
            f"into {table_name}"
        )


print(
    "\nDatabase created at:"
)

print(database_path)

In [ ]:
index_statements = [
    """
    CREATE UNIQUE INDEX IF NOT EXISTS
    idx_customers_customerid
    ON customers(customerid);
    """,

    """
    CREATE UNIQUE INDEX IF NOT EXISTS
    idx_services_customerid
    ON customer_services(customerid);
    """,

    """
    CREATE UNIQUE INDEX IF NOT EXISTS
    idx_billing_customerid
    ON customer_billing(customerid);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_predictions_customerid
    ON churn_predictions(customerid);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_predictions_probability
    ON churn_predictions(churn_probability);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_predictions_campaign
    ON churn_predictions(campaign_eligible);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_predictions_risk
    ON churn_predictions(business_risk_segment);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_predictions_priority
    ON churn_predictions(priority_score);
    """
]


if "customer_explanations" in sql_tables:

    index_statements.append(
        """
        CREATE INDEX IF NOT EXISTS
        idx_explanations_customerid
        ON customer_explanations(customerid);
        """
    )


with sqlite3.connect(
    database_path
) as connection:

    for statement in index_statements:

        connection.execute(statement)


    connection.commit()


print("Database indexes created.")

In [ ]:
retention_actions_sql = """
CREATE TABLE IF NOT EXISTS retention_actions
(
    action_id INTEGER PRIMARY KEY AUTOINCREMENT,

    customerid TEXT NOT NULL,

    action_type TEXT,

    offer_name TEXT,

    campaign_status TEXT DEFAULT 'Pending',

    contact_date TEXT,

    accepted_offer INTEGER,

    actual_outcome TEXT,

    notes TEXT,

    created_at_utc TEXT DEFAULT CURRENT_TIMESTAMP
);
"""


with sqlite3.connect(
    database_path
) as connection:

    connection.execute(
        retention_actions_sql
    )

    connection.commit()


print(
    "Retention-actions table created."
)

In [ ]:
customer_risk_view_sql = """
DROP VIEW IF EXISTS vw_customer_risk;

CREATE VIEW vw_customer_risk AS

SELECT
    c.customerid,
    c.gender,
    c.senior_citizen,
    c.partner,
    c.dependents,
    c.city,

    b.tenure_months,
    b.contract,
    b.payment_method,
    b.monthly_charges,
    b.total_charges,
    b.cltv,

    p.actual_churn,
    p.churn_probability,
    p.analytical_predicted_churn,
    p.campaign_eligible,
    p.high_priority_customer,
    p.business_risk_segment,
    p.priority_score,
    p.expected_monthly_value_at_risk,
    p.campaign_rank,
    p.model_version,
    p.prediction_source,

    r.top_risk_reasons

FROM customers AS c

LEFT JOIN customer_billing AS b
    ON c.customerid = b.customerid

LEFT JOIN churn_predictions AS p
    ON c.customerid = p.customerid

LEFT JOIN customer_reason_summary AS r
    ON c.customerid = r.customerid;
"""


with sqlite3.connect(
    database_path
) as connection:

    connection.executescript(
        customer_risk_view_sql
    )

    connection.commit()


print(
    "vw_customer_risk created."
)

In [ ]:
campaign_queue_view_sql = """
DROP VIEW IF EXISTS vw_campaign_queue;

CREATE VIEW vw_campaign_queue AS

SELECT
    *

FROM vw_customer_risk

WHERE campaign_eligible = 1

ORDER BY
    campaign_rank ASC,
    priority_score DESC;
"""


with sqlite3.connect(
    database_path
) as connection:

    connection.executescript(
        campaign_queue_view_sql
    )

    connection.commit()


print(
    "vw_campaign_queue created."
)

In [ ]:
summary_views_sql = """
DROP VIEW IF EXISTS vw_contract_risk_summary;

CREATE VIEW vw_contract_risk_summary AS

SELECT
    contract,

    COUNT(*) AS customers,

    AVG(actual_churn) AS actual_churn_rate,

    AVG(churn_probability)
        AS average_churn_probability,

    SUM(campaign_eligible)
        AS campaign_customers,

    AVG(campaign_eligible)
        AS campaign_customer_rate,

    SUM(cltv)
        AS total_cltv,

    SUM(
        CASE
            WHEN campaign_eligible = 1
            THEN cltv
            ELSE 0
        END
    ) AS campaign_cltv

FROM vw_customer_risk

GROUP BY contract;


DROP VIEW IF EXISTS vw_city_risk_summary;

CREATE VIEW vw_city_risk_summary AS

SELECT
    city,

    COUNT(*) AS customers,

    AVG(actual_churn) AS actual_churn_rate,

    AVG(churn_probability)
        AS average_churn_probability,

    SUM(campaign_eligible)
        AS campaign_customers,

    SUM(
        CASE
            WHEN campaign_eligible = 1
            THEN cltv
            ELSE 0
        END
    ) AS campaign_cltv

FROM vw_customer_risk

GROUP BY city;
"""


with sqlite3.connect(
    database_path
) as connection:

    connection.executescript(
        summary_views_sql
    )

    connection.commit()


print(
    "Contract and city views created."
)

In [ ]:
with sqlite3.connect(
    database_path
) as connection:

    database_objects = pd.read_sql_query(
        """
        SELECT
            type,
            name

        FROM sqlite_master

        WHERE type IN ('table', 'view')

        ORDER BY type, name;
        """,

        connection
    )


display(database_objects)

In [ ]:
table_names = [
    name

    for name in database_objects.loc[
        database_objects[
            "type"
        ] == "table",
        "name"
    ]

    if not name.startswith(
        "sqlite_"
    )
]


row_count_results = []


with sqlite3.connect(
    database_path
) as connection:

    for table_name in table_names:

        count = connection.execute(
            f"""
            SELECT COUNT(*)
            FROM "{table_name}";
            """
        ).fetchone()[0]


        row_count_results.append(
            {
                "table": table_name,
                "rows": count
            }
        )


table_row_counts = pd.DataFrame(
    row_count_results
)


display(table_row_counts)

In [ ]:
with sqlite3.connect(
    database_path
) as connection:

    top_campaign_customers = (
        pd.read_sql_query(
            """
            SELECT
                campaign_rank,
                customerid,
                churn_probability,
                business_risk_segment,
                cltv,
                priority_score,
                contract,
                tenure_months,
                monthly_charges,
                top_risk_reasons

            FROM vw_campaign_queue

            LIMIT 20;
            """,

            connection
        )
    )


display(top_campaign_customers)

In [ ]:
with sqlite3.connect(
    database_path
) as connection:

    contract_risk = pd.read_sql_query(
        """
        SELECT
            contract,
            customers,

            ROUND(
                actual_churn_rate * 100,
                2
            ) AS actual_churn_percent,

            ROUND(
                average_churn_probability * 100,
                2
            ) AS average_predicted_risk_percent,

            campaign_customers,

            ROUND(
                campaign_customer_rate * 100,
                2
            ) AS campaign_customer_percent,

            ROUND(
                campaign_cltv,
                2
            ) AS campaign_cltv

        FROM vw_contract_risk_summary

        ORDER BY
            average_churn_probability DESC;
        """,

        connection
    )


display(contract_risk)

In [ ]:
with sqlite3.connect(
    database_path
) as connection:

    high_value_risk = pd.read_sql_query(
        """
        SELECT
            customerid,
            churn_probability,
            cltv,
            priority_score,
            business_risk_segment,
            contract,
            tenure_months,
            top_risk_reasons

        FROM vw_customer_risk

        WHERE
            campaign_eligible = 1
            AND cltv IS NOT NULL

        ORDER BY
            priority_score DESC

        LIMIT 25;
        """,

        connection
    )


display(high_value_risk)

In [ ]:
if "customer_explanations" in sql_tables:

    with sqlite3.connect(
        database_path
    ) as connection:

        common_risk_reasons = pd.read_sql_query(
            """
            SELECT
                feature,
                feature_value,

                COUNT(*) AS occurrences,

                ROUND(
                    AVG(
                        shap_log_odds_contribution
                    ),
                    4
                ) AS average_shap_contribution

            FROM customer_explanations

            WHERE driver_type =
                'Raises churn risk'

            GROUP BY
                feature,
                feature_value

            ORDER BY
                occurrences DESC,
                average_shap_contribution DESC

            LIMIT 25;
            """,

            connection
        )


    display(common_risk_reasons)

In [ ]:
database_summary_path = (
    database_directory
    / "database_summary.csv"
)


table_row_counts.to_csv(
    database_summary_path,
    index=False
)


print("=" * 65)

print(
    "NOTEBOOK 7 — THRESHOLD POLICY AND SQL COMPLETE"
)

print("=" * 65)

print(
    f"Analytical threshold: "
    f"{analytical_threshold:.4f}"
)

print(
    f"Campaign capacity: "
    f"{CAMPAIGN_CAPACITY_RATE:.0%}"
)

print(
    f"Campaign threshold: "
    f"{campaign_threshold:.4f}"
)

print(
    f"High-risk threshold: "
    f"{high_risk_threshold:.4f}"
)

print(
    "\nSQLite database:"
)

print(database_path)

print(
    "\nBusiness prediction file:"
)

print(business_prediction_path)

print(
    "\nMain SQL view:"
)

print("vw_customer_risk")

print(
    "\nCampaign view:"
)

print("vw_campaign_queue")

In [ ]:
import sqlite3
import pandas as pd

powerbi_directory = (
    project_root
    / "reports"
    / "powerbi"
)

powerbi_directory.mkdir(
    parents=True,
    exist_ok=True
)


with sqlite3.connect(database_path) as connection:

    customer_risk_powerbi = pd.read_sql_query(
        """
        SELECT *
        FROM vw_customer_risk;
        """,
        connection
    )

    contract_summary_powerbi = pd.read_sql_query(
        """
        SELECT *
        FROM vw_contract_risk_summary;
        """,
        connection
    )

    city_summary_powerbi = pd.read_sql_query(
        """
        SELECT *
        FROM vw_city_risk_summary;
        """,
        connection
    )


customer_risk_powerbi.to_csv(
    powerbi_directory
    / "customer_risk_powerbi.csv",
    index=False
)

contract_summary_powerbi.to_csv(
    powerbi_directory
    / "contract_risk_powerbi.csv",
    index=False
)

city_summary_powerbi.to_csv(
    powerbi_directory
    / "city_risk_powerbi.csv",
    index=False
)


print("Power BI files exported:")
print(powerbi_directory)